In [ ]:
import tqdm.auto as tqdm

import drjit as dr
import matplotlib as mpl
import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

sns.set_theme(style="ticks", rc={"image.cmap": "rocket"})
mi.set_variant("scalar_mono_polarized_double")
# mi.set_variant("scalar_mono_double")


In [ ]:
# Load reference data
def load_reference(path):
    # Load the data
    da = xr.load_dataset(path).stokes

    # Add zenith angle coordinates
    da = da.assign_coords(
        {
            "theta": (
                "mu",
                np.rad2deg(np.arccos(da.mu.values)),
                {"long_name": "Viewing zenith angle", "units": "degree"},
            ),
            "theta0": (
                "mu0",
                np.rad2deg(np.arccos(da.mu0.values)),
                {"long_name": "Viewing zenith angle", "units": "degree"},
            ),
        }
    )

    # Split Stokes parameter dimension into variables
    vars = {
        x: da.sel(parameter=x, drop=True).rename(x)
        for x in da.coords["parameter"].values
    }

    # Swap conventions for Q component
    vars["Q"] *= -1

    # Compute degree of polarization
    vars["DLP"] = 100 * np.sqrt(vars["Q"] ** 2 + vars["U"] ** 2) / vars["I"]

    return xr.Dataset(data_vars=vars)


natraj_2009 = load_reference("natraj_2009.nc")
display(natraj_2009)
natraj_2012 = load_reference("natraj_2012.nc")
# display(natraj_2012)

# natraj_2012.sel(parameter="I", direction="DN", tau=1.0, albedo=0.8, mu0=0.14).dropna(
#     "mu"
# ).squeeze().plot()


In [ ]:
def sph_to_dir(theta, phi):
    """Map spherical to Euclidean coordinates."""
    st = np.sin(theta)
    ct = np.cos(theta)
    sp = np.sin(phi)
    cp = np.cos(phi)
    return np.array([cp * st, sp * st, ct])


def directions(mus, phis):
    """
    Generate a sequence of directions from cosine zenith and azimuth angle
    sequences.

    Parameters
    ----------
    mus : array-like
        Cosine of the zenith angle.

    phis : array-like
        Relative azimuth angle in degree.

    Returns
    -------
    directions : ndarray
        An array with shape (3, n) holding the (x, y, z) of directions defined
        by the Cartesian product of (mus, phis).
    """
    # Compute the Cartesian product of the zenith and azimuth sequences
    thetas = np.arccos(mus)
    phis = np.deg2rad(phis)
    angles = np.meshgrid(thetas, phis)

    # Turn it into a sequence of directions
    return sph_to_dir(angles[0].ravel(), angles[1].ravel())


def film_dict(height, width):
    return {
        "type": "hdrfilm",
        "width": height,
        "height": width,
        "pixel_format": "luminance",
        "component_format": "float32",
        "rfilter": {"type": "box"},
    }


def sampler_dict():
    return {"type": "independent"}


def integrator_dict(variant):
    # inner = {"type": "volpatherd"}
    # inner = {"type": "volpathaos"}
    # inner = {"type": "volpathsalesin"}
    inner = {"type": "volpath"}
    return {"type": "stokes", "integrator": inner} if "polarized" in variant else inner


def batch_radiancemeter_dict(thetas, phis, target=(0, 0, 0), radius=1.0):
    thetas = np.reshape(thetas, (-1,))
    phis = np.reshape(phis, (-1,))
    result = {"type": "batch"}

    for i, (theta, phi) in enumerate(zip(thetas, phis)):
        o = mi.ScalarPoint3f(target)
        p = target + mi.ScalarPoint3f(radius * sph_to_dir(theta, phi))
        d = p - o
        y = mi.ScalarVector3f([0, 1, 0])
        z = mi.ScalarVector3f([0, 0, 1])
        r = dr.normalize(dr.cross(z, d)) if theta != 0.0 else y
        l = dr.normalize(dr.cross(d, r))

        result[f"sensor_{i}"] = {
            "type": "radiancemeter",
            "to_world": mi.ScalarTransform4f.look_at(origin=p, target=o, up=l),
            "film": film_dict(1, 1),
        }

    result["film"] = film_dict(i + 1, 1)

    return result


def scene_dict(
    mu: np.typing.ArrayLike | None = None,
    phi: np.typing.ArrayLike | None = None,
    mu0: float = 1.0,
    tau: float = 1.0,
    albedo: float = 0.5,
    rayleigh: str = "rayleigh_polarized",
):
    """
    Parameters
    ----------
    mu : array-like
        Cosine of viewing zenith angles.

    phi : array-like
        Relative azimuth angle in degree.

    mu0 : float
        Cosine of illumination zenith angle.

    tau : float
        Optical thickness of the Rayleigh scattering layer.

    rho : float
        Surface reflectance.

    rayleigh : str
        Rayleigh plugin implementation to be used.
    """
    if mu is None:
        zeniths = np.arange(0, 86, 5)
        mu = np.cos(np.deg2rad(zeniths))

    if phi is None:
        phi = np.array([0.0, 180.0])

    thetas, phis = np.meshgrid(np.arccos(mu), np.deg2rad(phi))

    shape_scale = mi.ScalarTransform4f.scale([1e3, 1e3, 1])
    sensor_directions = -directions(mu, phi).T
    illumination_direction = -directions(mu0, 180.0).squeeze()
    surface_reflectance = albedo
    sigma_t = tau

    result = {
        "type": "scene",
        "illumination": {
            "type": "directional",
            "direction": illumination_direction,
            "irradiance": 1.0 * np.pi,
        },
        "medium_atmosphere": {
            "type": "homogeneous",
            "sigma_t": sigma_t,
            "albedo": 1.0,
            "phase_function": {"type": rayleigh},
        },
        "shape_atmosphere": {
            "type": "cube",
            "to_world": shape_scale,
            "interior": {"type": "ref", "id": "medium_atmosphere"},
            "bsdf": {"type": "null"},
        },
        "surface_shape": {
            "type": "rectangle",
            "to_world": shape_scale,
            # "bsdf": {"type": "diffuse", "reflectance": {"type": "checkerboard"}},
            "bsdf": {"type": "diffuse", "reflectance": surface_reflectance},
            # "bsdf": {"type": "roughplastic"},
            # "bsdf": {"type": "rpv"},
        },
        "sensor_batch": batch_radiancemeter_dict(
            thetas, phis, target=[0, 0, 1], radius=1
        ),
        "sensor_camera_toa": {
            "type": "perspective",
            "to_world": mi.ScalarTransform4f.look_at(
                origin=[0, 0, 10],
                target=[0, 0, 0],
                up=[0, 1, 0],
            ),
            "far_clip": 1e10,
            "film": film_dict(320, 240),
            "sampler": sampler_dict(),
        },
        "sensor_camera_specular": {
            "type": "perspective",
            "to_world": mi.ScalarTransform4f.look_at(
                origin=2e3
                * mi.ScalarPoint3f(
                    [
                        illumination_direction[0],
                        0.0,
                        -illumination_direction[2],
                    ]
                ),
                target=[0, 0, 0],
                up=[0, 0, 1],
            ),
            "far_clip": 1e10,
            "film": film_dict(320, 240),
            "sampler": sampler_dict(),
        },
        "sensor_camera_oblique": {
            "type": "perspective",
            "to_world": mi.ScalarTransform4f.look_at(
                origin=1e3 * mi.ScalarPoint3f([2, 2, 1]),
                target=[0, 0, 0],
                up=[0, 0, 1],
            ),
            "far_clip": 1e10,
            "film": film_dict(320, 240),
            "sampler": sampler_dict(),
        },
        "sensor_mdistant_toa": {
            "type": "mdistant",
            "directions": ",".join(f"{x[0]},{x[1]},{x[2]}" for x in sensor_directions),
            "target": [0, 0, 0],
            "film": film_dict(len(sensor_directions), 1),
            "sampler": sampler_dict(),
        },
        "integrator": integrator_dict(mi.variant()),
    }

    return result


In [ ]:
scene = mi.load_dict(scene_dict(mu0=0.75, tau=1.0))
sensors = {x.id(): x for x in scene.sensors()}

for sensor_id, sensor in sensors.items():
    if "mdistant" in sensor_id:
        spp = 4**8
    elif "batch" in sensor_id:
        spp = 4**8
    else:
        spp = 4**2

    print(f"Rendering '{sensor_id}'")
    result = mi.render(scene, sensor=sensor, spp=spp)


In [ ]:
import os
from collections import defaultdict

FIGSIZE = {
    "camera": (4 * 5, 4),
    "mdistant": (4 * 5, 3),
    "batch": (4 * 5, 3),
}

CMAP = defaultdict(
    lambda: sns.color_palette("rocket", as_cmap=True),
    {
        "S1": sns.color_palette("icefire", as_cmap=True),
        "S2": sns.color_palette("icefire", as_cmap=True),
        "S3": sns.color_palette("icefire", as_cmap=True),
    },
)

VLIM = defaultdict(
    lambda: [None, None],
    {
        "<root>": [0, 1.5],
        "S0": [0, 1.5],
        "S1": [-0.25, 0.25],
        "S2": [-0.15, 0.15],
    },
)

TITLE = {
    "<root>": "Radiance",
    "S0": "I",
    "S1": "Q",
    "S2": "U",
    "S3": "V",
}


def savefig(fig, path, **kwargs):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, **kwargs)


for sensor_id in [
    "sensor_camera_oblique",
    "sensor_camera_specular",
    "sensor_camera_toa",
    "sensor_mdistant_toa",
    "sensor_batch",
]:
    sensor = sensors[sensor_id]
    try:
        images = dict(sensor.film().bitmap().split())
    except RuntimeError:
        continue

    sensor_id = sensor.id()
    if "camera" in sensor_id:
        sensor_type = "camera"
    elif "mdistant" in sensor_id:
        sensor_type = "mdistant"
    elif "batch" in sensor_id:
        sensor_type = "batch"
    else:
        raise ValueError

    fig, axs = plt.subplots(1, 5, figsize=FIGSIZE[sensor_type], layout="constrained")
    fig.suptitle(sensor_id)

    for ax, (channel, img) in zip(axs, images.items()):
        ax.set_title(TITLE[channel])
        img = np.atleast_3d(img)[:, :, 0]

        if sensor_type == "camera":
            vlim = vlim = VLIM[channel]
            mappable = ax.imshow(img, cmap=CMAP[channel], vmin=vlim[0], vmax=vlim[1])
            plt.colorbar(mappable, ax=ax, orientation="horizontal")

        elif sensor_type in {"mdistant", "batch"}:
            y = img.squeeze()
            ax.plot(np.concatenate((y[len(y) // 2-1 :: -1], y[len(y) // 2 :])))

    savefig(fig, f"validation/validation_{sensor_id}_{mi.variant()}.pdf")
    plt.show()
    plt.close()


In [ ]:
mus = natraj_2009.mu.values
phis = natraj_2009.phi.values
param_space_natraj_2009 = pd.MultiIndex.from_product(
    [
        natraj_2009.mu0.values,
        natraj_2009.tau.values,
        natraj_2009.albedo.values,
    ],
    names=["mu0", "tau", "albedo"],
)


In [ ]:
# mi.set_variant("scalar_mono_double")
mi.set_variant("scalar_mono_polarized_double")


scenes = {}
sel = dict(direction="UP", tau=0.5, albedo=0.8, mu0=1.0)
# sel = dict(direction="UP", tau=0.5, albedo=0.8, mu0=0.6)
# sel = dict(direction="UP", tau=0.5, albedo=0.25, mu0=0.8)
# sel = dict(direction="UP", tau=0.05, albedo=0.0, mu0=0.1)
# sel = dict(direction="UP", tau=0.02, albedo=0.0, mu0=0.1)

# for mu0, tau, albedo in tqdm.tqdm(param_space_natraj_2009):  # Full space
for mu0, tau, albedo in tqdm.tqdm([[sel["mu0"], sel["tau"], sel["albedo"]]]):  # Single configuration (debug)
    scene = mi.load_dict(scene_dict(mus, phis, mu0, tau, albedo, rayleigh="rayleigh_polarized"))
    sensors = {x.id(): x for x in scene.sensors()}

    for sensor_id, sensor in sensors.items():
        if "mdistant" in sensor_id or "batch" in sensor_id:
            spp = 4 ** 4  # Data structure debug
            spp = 4 ** 6  # Array layout debug
            spp = 4 ** 8  # Decent estimate
            # spp = 4 ** 10  # Production
        else:
            spp = 4 ** 2
            continue  # Just ignore the cameras

        print(f"Rendering '{sensor_id}'")
        mi.render(scene, sensor=sensor, spp=spp)
        scenes[mu0, tau, albedo] = scene

    break


In [ ]:
STOKES_COMPONENTS = (
    {"S0": "I", "S1": "Q", "S2": "U", "S3": "V"}
    if "polarized" in mi.variant()
    else {"<root>": "I"}
)
STOKES_PARAMETERS = ["I", "Q", "U"] if "polarized" in mi.variant() else ["I"]

results = xr.full_like(natraj_2009, np.nan)

for (mu0, tau, albedo), scene in scenes.items():
    images = {}
    sensor = scene.sensors()[0]
    for k, v in sensor.film().bitmap().split():
        if k in STOKES_COMPONENTS:
            v = np.atleast_3d(v)
            # print(f"{v.shape = }")
            # print(f"{v = }")
            images[STOKES_COMPONENTS[k]] = v[0, :, 0].reshape((len(phis), len(mus))).T
    # print(images)

    for parameter in STOKES_PARAMETERS:
        results[parameter].sel(direction="UP", tau=tau, albedo=albedo, mu0=mu0).values[
            :, :
        ] = images[parameter]

results["DLP"] = 100 * np.sqrt(results["Q"] ** 2 + results["U"] ** 2) / results["I"]
results["DLP"].attrs = {"units": "percent"}

x = results.copy()
for dim in ["direction", "tau", "albedo", "mu0"]:
    x = x.dropna(dim, how="all")
x = x.squeeze()
x


In [ ]:
CMAP_PARAMS = defaultdict(
    lambda: (sns.color_palette("rocket", as_cmap=True), None),
    {
        ("I", "reldiff"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("Q", "mitsuba"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("Q", "natraj"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("Q", "ratio"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("Q", "reldiff"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("U", "mitsuba"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("U", "natraj"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("U", "ratio"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("U", "reldiff"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
        ("DLP", "reldiff"): (
            sns.color_palette("icefire", as_cmap=True),
            mpl.colors.CenteredNorm,
        ),
    },
)

params = ["I", "DLP", "Q", "U"]
datasets = [
    (
        "mitsuba",
        results,
        "Mitsuba",
    ),
    (
        "natraj",
        natraj_2009,
        "Reference (Natraj, 2009)",
    ),
    (
        "absdiffdiff",
        results - natraj_2009,
        "Absolute difference",
    ),
    (
        "reldiff",
        ((results - natraj_2009) / natraj_2009 * 100),
        "Relative difference [%]",
    ),
    (
        "ratio",
        (results / natraj_2009),
        "Ratio",
    ),
]

nrows = len(params)
ncols = len(datasets)

fig, axs = plt.subplots(
    nrows,
    ncols,
    figsize=(3.5 * ncols, 2.5 * nrows),
    layout="constrained",
    sharex=True,
    sharey=True,
)

cbar_kwargs = {"label": "", "orientation": "vertical"}

for irow, stokes_param in enumerate(params):
    for icol, (ds_id, ds, title) in enumerate(datasets):
        # print(f"Processing {stokes_param}, {ds_id}")
        ax = axs[irow, icol]
        da = ds[stokes_param].sel(**sel)
        cmap, norm = CMAP_PARAMS[stokes_param, ds_id]

        if (
            (norm is None)
            or np.all(da.values == 0.0)
            or np.all(np.isnan(da.values))
            or np.all(np.isinf(da.values))
        ):
            norm = None
        else:
            norm = norm()

        da.plot.imshow(
            ax=ax,
            cmap=cmap,
            cbar_kwargs=cbar_kwargs,
            norm=norm,
            robust=True,
        )

        if irow == 0:
            ax.set_title(title)
        else:
            ax.set_title("")

        if irow == len(params) - 1:
            ax.set_xlabel("Relative azimuth [°]")
            ax.set_xticks([0, 90, 180])
        else:
            ax.set_xlabel("")

        if icol == 0:
            ax.set_ylabel(f"{stokes_param}\nCosine zenith [—]")
        else:
            ax.set_ylabel("")


plt.show()


In [ ]:
fig, axs = plt.subplots(
    4,
    4,
    figsize=(4 * 4.5, 3 * 3),
    layout="constrained",
    sharex=True,
)

cbar_kwargs = {"label": "", "orientation": "horizontal"}
phi = 30

for irow, stokes_param in enumerate(["I", "Q", "U"]):
    for icol, (ds_id, ds, title) in enumerate(
        [
            (
                "mitsuba",
                results,
                "Mitsuba",
            ),
            (
                "natraj",
                natraj_2009,
                "Reference (Natraj, 2009)",
            ),
            (
                "ratio",
                (results / natraj_2009),
                "Ratio",
            ),
            (
                "reldiff",
                ((results - natraj_2009) / natraj_2009 * 100),
                "Relative difference [%]",
            ),
        ]
    ):
        ax = axs[irow, icol]
        ds[stokes_param].sel(**sel, phi=phi).plot.line(ax=ax, x="theta")

        if irow == 0:
            ax.set_title(title)
        else:
            ax.set_title("")

        if irow == 2:
            ax.set_xlabel("Zenith [°]")
        else:
            ax.set_xlabel("")

        if icol == 0:
            ax.set_ylabel(f"{stokes_param}\nCosine zenith [—]")
        else:
            ax.set_ylabel("")


fig.suptitle(
    " $\cdot$ ".join(
        [
            f"{sel['direction']}",
            f"$τ$ = {sel['tau']}",
            f"$A$ = {sel['albedo']}",
            f"$μ_0$ = {sel['mu0']}",
            f"$φ$ = {phi}°",
        ]
    )
)

plt.show()
